In [0]:
from pyspark.sql import functions as F, Window

CAT = "workspace"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CAT}.gold")

def salvar(df, nome):
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{CAT}.gold.{nome}"))

In [0]:
# chaves substitutas geradas com row_number ordenado pela chave natural, assim são as mesmas a cada execução do job. a chave natural fica na dim_movies

# metadados de cada filme
info = spark.table(f"{CAT}.silver.tb_info_filmes")
dim_movies = (info
    .withColumn("sk_movie_id", F.row_number().over(Window.orderBy("id_filme")).cast("bigint"))
    .select("sk_movie_id", "id_filme", "titulo", "data_lancamento", "ano_lancamento",
            "duracao_minutos", "idioma_original", "status_filme", "sinopse"))
salvar(dim_movies, "dim_movies")

# catálogo único de gêneros
gen = spark.table(f"{CAT}.silver.tb_generos").select(F.col("genero").alias("nome_genero")).distinct()
dim_genres = gen.withColumn("sk_genre_id", F.row_number().over(Window.orderBy("nome_genero")).cast("bigint")) \
                .select("sk_genre_id", "nome_genero")
salvar(dim_genres, "dim_genres")

# só pessoas. a mesma pessoa em papéis diferentes vira uma linha por papel
pe = spark.table(f"{CAT}.silver.tb_pessoas_empresas")
pessoas = (pe.filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
             .select(F.col("nome_entidade").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa")).distinct())
dim_people = pessoas.withColumn("sk_person_id", F.row_number().over(Window.orderBy("tipo_pessoa", "nome_pessoa")).cast("bigint")) \
                    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
salvar(dim_people, "dim_people")

# catálogo único de produtoras
emp = (pe.filter(F.col("tipo_entidade") == "Produtora")
         .select(F.col("nome_entidade").alias("nome_produtora")).distinct())
dim_companies = emp.withColumn("sk_company_id", F.row_number().over(Window.orderBy("nome_produtora")).cast("bigint")) \
                   .select("sk_company_id", "nome_produtora")
salvar(dim_companies, "dim_companies")

# avaliações resumidas por filme
dm = spark.table(f"{CAT}.gold.dim_movies").select("sk_movie_id", "id_filme")
rev = (spark.table(f"{CAT}.silver.tb_avaliacoes_usuarios")
    .groupBy("id_filme")
    .agg(F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
         F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios")))
dim_reviews = (rev.join(dm, "id_filme")
    .withColumn("sk_review_id", F.row_number().over(Window.orderBy("sk_movie_id")).cast("bigint"))
    .select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios"))
salvar(dim_reviews, "dim_reviews")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# 1 linha por filme lançado. financeiro e métricas têm 1 linha por filme na silver, então os joins não duplicam o grão. left join pra não perder filme sem dado financeiro
dm = spark.table(f"{CAT}.gold.dim_movies")
fin = spark.table(f"{CAT}.silver.tb_financeiro_filmes")
met = spark.table(f"{CAT}.silver.tb_metricas_engajamento")

fact = (dm.filter(F.col("status_filme") == "Lançado").select("sk_movie_id", "id_filme")
    .join(fin, "id_filme", "left")
    .join(met, "id_filme", "left")
    .select("sk_movie_id",
            F.col("orcamento_usd").cast("decimal(18,2)"), F.col("receita_usd").cast("decimal(18,2)"),
            F.col("lucro_usd").cast("decimal(18,2)"),
            F.col("orcamento_brl").cast("decimal(18,2)"), F.col("receita_brl").cast("decimal(18,2)"),
            F.col("lucro_brl").cast("decimal(18,2)"),
            F.col("popularidade").cast("double"), F.col("nota_media_tmdb").cast("double"),
            F.col("qtd_votos_tmdb").cast("int"), F.col("nota_media_imdb").cast("double"),
            F.col("qtd_votos_imdb").cast("int")))
salvar(fact, "fact_movies_performance")

In [0]:
# tabelas-ponte ligam filme a gênero/pessoa/produtora (N:N) sem duplicar a fato
dm = spark.table(f"{CAT}.gold.dim_movies").select("sk_movie_id", "id_filme")
pe = spark.table(f"{CAT}.silver.tb_pessoas_empresas")

dg = spark.table(f"{CAT}.gold.dim_genres")
salvar(spark.table(f"{CAT}.silver.tb_generos")
    .join(dm, "id_filme").join(dg, F.col("genero") == dg["nome_genero"])
    .select("sk_movie_id", "sk_genre_id").distinct(), "bridge_movie_genre")

dp = spark.table(f"{CAT}.gold.dim_people")
salvar(pe.filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .join(dm, "id_filme")
    .join(dp, (pe["nome_entidade"] == dp["nome_pessoa"]) & (pe["tipo_entidade"] == dp["tipo_pessoa"]))
    .select("sk_movie_id", "sk_person_id").distinct(), "bridge_movie_person")

dc = spark.table(f"{CAT}.gold.dim_companies")
salvar(pe.filter(F.col("tipo_entidade") == "Produtora")
    .join(dm, "id_filme")
    .join(dc, pe["nome_entidade"] == dc["nome_produtora"])
    .select("sk_movie_id", "sk_company_id").distinct(), "bridge_movie_company")

In [0]:
for t in ["dim_movies", "dim_genres", "dim_people", "dim_companies", "dim_reviews",
          "fact_movies_performance", "bridge_movie_genre", "bridge_movie_person", "bridge_movie_company"]:
    print(t, spark.table(f"{CAT}.gold.{t}").count())

f = spark.table(f"{CAT}.gold.fact_movies_performance")
print("fato: linhas", f.count(), "| filmes únicos", f.select("sk_movie_id").distinct().count())
f.printSchema()

dim_movies 97879
dim_genres 19
dim_people 419240
dim_companies 45753
dim_reviews 27303
fact_movies_performance 96522
bridge_movie_genre 140728
bridge_movie_person 763308
bridge_movie_company 119209
fato: linhas 96522 | filmes únicos 96522
root
 |-- sk_movie_id: long (nullable = true)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- receita_usd: decimal(18,2) (nullable = true)
 |-- lucro_usd: decimal(18,2) (nullable = true)
 |-- orcamento_brl: decimal(18,2) (nullable = true)
 |-- receita_brl: decimal(18,2) (nullable = true)
 |-- lucro_brl: decimal(18,2) (nullable = true)
 |-- popularidade: double (nullable = true)
 |-- nota_media_tmdb: double (nullable = true)
 |-- qtd_votos_tmdb: integer (nullable = true)
 |-- nota_media_imdb: double (nullable = true)
 |-- qtd_votos_imdb: integer (nullable = true)



In [ ]:
# 1 linha por filme, mesmo os sem financeiro, elenco ou diretor. concat vira NULL se um campo for nulo, então cada campo passa por coalesce com texto padrão
dm   = spark.table(f"{CAT}.gold.dim_movies")
fact = spark.table(f"{CAT}.gold.fact_movies_performance")
dp   = spark.table(f"{CAT}.gold.dim_people")
bmp  = spark.table(f"{CAT}.gold.bridge_movie_person")

pp = bmp.join(dp, "sk_person_id")

# até 5 atores por filme, em ordem alfabética
atores = (pp.filter(F.col("tipo_pessoa") == "Ator")
    .withColumn("rn", F.row_number().over(Window.partitionBy("sk_movie_id").orderBy("nome_pessoa")))
    .filter("rn <= 5")
    .groupBy("sk_movie_id")
    .agg(F.array_join(F.sort_array(F.collect_list("nome_pessoa")), ", ").alias("atores")))

diretores = (pp.filter(F.col("tipo_pessoa") == "Diretor")
    .groupBy("sk_movie_id")
    .agg(F.array_join(F.sort_array(F.collect_list("nome_pessoa")), ", ").alias("diretores")))

# nulo, vazio ou só espaço cai no texto padrão
def ou(coluna, fallback):
    v = F.trim(coluna.cast("string"))
    return F.coalesce(F.when(v != "", v), F.lit(fallback))

def dinheiro(coluna):
    return F.coalesce(F.concat(F.lit("US$ "), F.format_number(coluna, 2)), F.lit("valor não informado"))

base = (dm.join(fact, "sk_movie_id", "left")
          .join(atores, "sk_movie_id", "left")
          .join(diretores, "sk_movie_id", "left"))

documento = F.concat(
    F.lit("O filme "), ou(F.col("titulo"), "sem título informado"),
    F.lit(", lançado no ano de "), ou(F.col("ano_lancamento"), "ano não informado"),
    F.lit(", faturou "), dinheiro(F.col("receita_usd")),
    F.lit(" e teve um custo de "), dinheiro(F.col("orcamento_usd")),
    F.lit(". Estrelado por "), ou(F.col("atores"), "elenco não informado"),
    F.lit(" e dirigido por "), ou(F.col("diretores"), "diretor não informado"),
    F.lit(", o filme possui a seguinte sinopse: "), ou(F.col("sinopse"), "sinopse não disponível"),
    F.lit("."))

genai = base.select(
    F.col("id_filme").alias("movie_id"),
    F.col("titulo").alias("title"),
    documento.alias("llm_context_document"))

salvar(genai, "gold_genai_movies_context")

In [ ]:
g = spark.table(f"{CAT}.gold.gold_genai_movies_context")
print("linhas:", g.count(), "| filmes na dim_movies:", spark.table(f"{CAT}.gold.dim_movies").count())
print("documentos nulos:", g.filter("llm_context_document is null").count())
g.select("movie_id", "title", "llm_context_document").show(3, False)

# filmes com fallback, prova que os nulos não derrubaram nenhuma linha
g.filter(F.col("llm_context_document").contains("não informado")).select("llm_context_document").show(3, False)

In [ ]:
fact = spark.table(f"{CAT}.gold.fact_movies_performance")
dm   = spark.table(f"{CAT}.gold.dim_movies")

# receita total em R$
display(fact.agg(F.sum("receita_brl").alias("receita_total_brl")))

# top 5 popularidade
display(fact.join(dm, "sk_movie_id")
            .select("titulo", "popularidade")
            .orderBy(F.col("popularidade").desc_nulls_last())
            .limit(5))

# filmes por gênero, do maior pro menor
display(spark.table(f"{CAT}.gold.bridge_movie_genre")
            .join(spark.table(f"{CAT}.gold.dim_genres"), "sk_genre_id")
            .groupBy("nome_genero").agg(F.countDistinct("sk_movie_id").alias("qtd_filmes"))
            .orderBy(F.col("qtd_filmes").desc()))

In [ ]:
# top 10 por receita com rank(): filmes empatados ficam na mesma posição
w = Window.orderBy(F.col("receita_usd").desc())
top10 = (fact.join(dm, "sk_movie_id")
    .filter(F.col("receita_usd").isNotNull())
    .withColumn("posicao_ranking", F.rank().over(w))
    .filter("posicao_ranking <= 10")
    .select("posicao_ranking", "titulo", "receita_usd", "receita_brl")
    .orderBy("posicao_ranking"))
display(top10)

In [ ]:
# data limite = lançamento mais recente da base, sem contar filme não lançado nem data futura
data_limite = (dm.filter((F.col("status_filme") == "Lançado") & (F.col("data_lancamento") <= F.current_date()))
                 .agg(F.max("data_lancamento")).first()[0])
print("data limite:", data_limite)

def ultimos_anos(anos):
    return (dm.filter((F.col("status_filme") == "Lançado")
                      & (F.col("data_lancamento") <= F.lit(data_limite))
                      & (F.col("data_lancamento") > F.add_months(F.lit(data_limite), -12 * anos)))
              .select("sk_movie_id"))

# ator com mais filmes nos últimos 2 anos (mostro o top 5 pra ver se tem empate)
display(ultimos_anos(2)
    .join(spark.table(f"{CAT}.gold.bridge_movie_person"), "sk_movie_id")
    .join(spark.table(f"{CAT}.gold.dim_people").filter(F.col("tipo_pessoa") == "Ator"), "sk_person_id")
    .groupBy("nome_pessoa").agg(F.countDistinct("sk_movie_id").alias("qtd_filmes"))
    .orderBy(F.col("qtd_filmes").desc(), "nome_pessoa")
    .limit(5))

# produtora com maior lucro nos últimos 5 anos
# só entra filme com orçamento e receita conhecidos, senão o lucro fica distorcido
display(ultimos_anos(5)
    .join(fact.filter(F.col("orcamento_usd").isNotNull() & F.col("receita_usd").isNotNull()), "sk_movie_id")
    .join(spark.table(f"{CAT}.gold.bridge_movie_company"), "sk_movie_id")
    .join(spark.table(f"{CAT}.gold.dim_companies"), "sk_company_id")
    .groupBy("nome_produtora")
    .agg(F.sum("lucro_usd").alias("lucro_usd"), F.sum("lucro_brl").alias("lucro_brl"))
    .orderBy(F.col("lucro_usd").desc())
    .limit(5))